# PathoLens AI — Inference and ExplainabilityNew, unexecuted notebook. It imports the same inference modules as the Streamlit application. Research prototype only; no diagnosis or validated localization.

## Model and project informationThe bundled E2 HED-augmented MobileNetV3-Small checkpoint leads the saved 50% test comparison. It was selected after inspecting test metrics, so those metrics are post-selection and not an unbiased final estimate. Input is RGB resized to 128×128 and ImageNet-normalized. Class names are read from `models/inference_config.json`. E2-specific calibration results are unavailable, so probabilities are raw softmax outputs.

In [ ]:
from pathlib import Pathfrom PIL import Imageimport matplotlib.pyplot as pltimport numpy as npimport torchfrom IPython.display import displayfrom src.model_loader import load_model, CONFIGfrom src.preprocessing import prepare_imagefrom src.prediction import predictfrom src.gradcam import generate_gradcamfrom src.bounding_boxes import activation_boxes, draw_boxesAPP_ROOT = Path.cwd()  # Start Jupyter with PathoLens_Application as the working directoryif not (APP_ROOT / "src").is_dir():    raise RuntimeError("Open this notebook with PathoLens_Application as the working directory.")model, device = load_model()print(CONFIG["experiment_id"], CONFIG["experiment_name"], "on", device)

## Load an uploaded or sample imageSet `IMAGE_PATH` to a JPG, JPEG, or PNG file. The original is retained for visual outputs; a resized and normalized copy is used for inference.

In [ ]:
IMAGE_PATH = Path("sample.jpg")  # Change to an image pathsource_image = Image.open(IMAGE_PATH)original, input_tensor = prepare_image(source_image)print("Original dimensions:", original.size, "model tensor:", tuple(input_tensor.shape))display(original)

## Prediction and nine-class probabilitiesSoftmax probabilities are computed from model logits. No E2-specific calibration settings were saved; report raw softmax probabilities only.

In [ ]:
result = predict(model, input_tensor, device)print("Predicted:", CONFIG["class_names"][result.class_index])print(f"Uncalibrated softmax confidence: {result.confidence:.4f}")fig, ax = plt.subplots(figsize=(9, 4))order = np.argsort(result.probabilities.cpu().numpy())ax.barh(np.array(CONFIG["class_names"])[order], result.probabilities.cpu().numpy()[order])ax.set_xlim(0, 1); ax.set_xlabel("Probability"); fig.tight_layout()

## Grad-CAM and activation-region boxesGrad-CAM targets the predicted class logit and final feature stage. The normalized map is thresholded, small connected regions are filtered, and coordinates are scaled independently to original width and height.

In [ ]:
heatmap = generate_gradcam(model, input_tensor, device, result.class_index)boxes = activation_boxes(heatmap, original.size)annotated = draw_boxes(original, boxes) if boxes else original.copy()print(f"Retained activation boxes: {len(boxes)}")if not boxes:    print("No meaningful activation component passed the filters; no box was created.")

## Three separate visual outputsThe heatmap is displayed independently. The final boxed image is drawn on a fresh copy of the original, without heatmap overlay.

In [ ]:
heat_l = Image.fromarray((heatmap * 255).clip(0, 255).astype("uint8"), mode="L").resize(original.size, Image.Resampling.BILINEAR)heat_rgb = Image.merge("RGB", (heat_l, Image.new("L", original.size, 0), Image.new("L", original.size, 0)))fig, axes = plt.subplots(1, 3, figsize=(15, 5))axes[0].imshow(original); axes[0].set_title("A · Original")axes[1].imshow(heat_rgb); axes[1].set_title("B · Grad-CAM heatmap")axes[2].imshow(annotated); axes[2].set_title("C · Original with activation boxes")for axis in axes: axis.axis("off")plt.tight_layout()

## Interpretation and limitationsClassification assigns one class to the whole input. Grad-CAM is a coarse class-associated localization map. It is not object detection or semantic segmentation. Boxes are approximate high-activation regions; they are not ground-truth lesion annotations, validated detections, or medical diagnoses. The model may be wrong, and heatmaps can highlight irrelevant context. Results in the packaged report come from a 50% pilot subset; E2-specific calibration results were unavailable, so probabilities are raw softmax and do not establish clinical safety or external generalization.